# Privacy gates and differentially private release

Publishing a pooled statistic exposes its contributors. `meta.privacy` and `meta.publish` gate a
release three ways, and each gate *refuses* rather than clips or pads:

1. **k-anonymity and dominance** (`PrivacyPolicy`, `check_cell`): a cell with fewer than `k`
   distinct contributors, or one where a single contributor (or the top `n`) holds too large a
   share of the absolute total, is `blocked`.
2. **An ε ledger** (`EpsilonLedger`, `charge`): an immutable budget under sequential
   composition; a charge that would exceed it returns `Blocked` and leaves the ledger unchanged.
3. **A mechanism** (`release`): the clipped mean with sensitivity `(hi − lo)/n`, noised by the
   Laplace mechanism (`b = Δ/ε`) or the analytic Gaussian mechanism (Balle & Wang 2018); the
   seed, clip bounds, sensitivity and scale are all on the `Release`.

In [ ]:
import numpy as np

from axiom.core import Blocked, Spec, Verdict
from axiom.meta import (
    Cell, EpsilonCharge, EpsilonLedger, EpsilonSplit, Mechanism, PrivacyPolicy, Release, StudyRecord,
    carry_forward, cell_from_records, charge, check_cell, contributor_totals, gaussian_sigma,
    gaussian_sigma_classical, jaccard_distance, laplace_scale, orthogonal_split, release,
)

## `Cell`, `cell_from_records`, `contributor_totals`

A `Cell` is the `(contributor, value)` pairs behind one publishable number; `cell_from_records`
builds one from `StudyRecord`s (their `contributor` and `estimate`). `contributor_totals` sums
`|value|` per contributor — the quantity the dominance rules look at.

In [ ]:
records = [
    StudyRecord(study=f"s{i}", contributor=c, quantity="elasticity", estimate=v, se=0.1, read="experiment", family="fertilizer")
    for i, (c, v) in enumerate([("c0", 0.42), ("c1", 0.55), ("c2", 0.31), ("c3", 0.67), ("c0", 0.48), ("c4", 0.39)])
]
cell: Cell = cell_from_records(records, name="fertilizer/elasticity")
print("n =", cell.n, "| contributors:", sorted(cell.contributors))
print("totals:", contributor_totals(cell))

## `PrivacyPolicy` and `check_cell`

`k` distinct contributors at least; no contributor above `dominance_p` of the absolute total; the
top `dominance_top_n` together below `dominance_top_p`. A cell one contributor short of `k` is
refused outright, and so is a dominated one.

In [ ]:
policy = PrivacyPolicy(k=3, dominance_p=0.5, dominance_top_n=2, dominance_top_p=0.8, epsilon_total=1.0)
ok: Verdict = check_cell(cell, policy)
print("full cell:", ok.status, "|", ok.route)
small = Cell(records=(("c0", 0.4), ("c1", 0.5)), name="k-1")
v = check_cell(small, policy)
print("k−1 cell: ", v.status, "|", v.reason)
dominated = Cell(records=(("c0", 5.0), ("c1", 0.5), ("c2", 0.4)), name="dominated")
v = check_cell(dominated, policy)
print("dominated:", v.status, "|", v.reason, "| rule:", v.route)
top2 = Cell(records=(("c0", 2.0), ("c1", 1.9), ("c2", 0.5), ("c3", 0.4)), name="top-2")
v = check_cell(top2, policy)
print("top-2:    ", v.status, "|", v.reason, "| rule:", v.route)

## `EpsilonLedger`, `EpsilonCharge`, `charge`

The ledger is immutable: `charge` returns a *new* ledger with an `EpsilonCharge` appended, or
`Blocked` leaving the original untouched. Charges compose sequentially, so `used + remaining`
is always the budget. Release ids are unique on a ledger.

In [ ]:
ledger = EpsilonLedger(budget=policy.epsilon_total)
mech: Mechanism = "laplace"
l1 = charge(ledger, release_id="q1", epsilon=0.3, mechanism=mech)
l2 = charge(l1, release_id="q2", epsilon=0.5, mechanism="gaussian", delta=1e-6)
assert isinstance(l1, EpsilonLedger) and isinstance(l2, EpsilonLedger)
line: EpsilonCharge = l2.charges[-1]
print("charges:", [(c.release_id, c.epsilon) for c in l2.charges], "| last:", line.mechanism, line.delta)
print(f"used {l2.used} + remaining {l2.remaining} = budget {l2.budget}  conserved: {l2.used + l2.remaining == l2.budget}")
over = charge(l2, release_id="q3", epsilon=0.3, mechanism="laplace")
print("over budget:", type(over).__name__, "-", over.reason)
print("original ledger untouched:", ledger.used == 0.0, "| duplicate id:", charge(l2, release_id="q1", epsilon=0.1, mechanism="laplace").reason)

## Noise calibration

`laplace_scale(Δ, ε) = Δ/ε`. For the Gaussian mechanism, `gaussian_sigma` is the analytic
calibration — the smallest `σ` with `Φ(Δ/(2σ) − εσ/Δ) − e^ε Φ(−Δ/(2σ) − εσ/Δ) ≤ δ` — which is
valid for every `ε > 0` and tighter than the classical `gaussian_sigma_classical`
(`Δ·sqrt(2 ln(1.25/δ))/ε`, valid only for `ε < 1`).

In [ ]:
sens = 1.0 / cell.n
print("laplace scale at ε=0.5:", laplace_scale(sens, 0.5))
for eps in (0.1, 0.5, 0.9):
    print(f"ε={eps}: analytic σ = {gaussian_sigma(sens, eps, 1e-6):.4f}   classical σ = {gaussian_sigma_classical(sens, eps, 1e-6):.4f}")
print("analytic still defined at ε=2:", round(gaussian_sigma(sens, 2.0, 1e-6), 4))

## `release`

The cell must clear the policy, the ledger must cover `epsilon`, and then the clipped mean is
noised. The `Release` records every input; its `interval` is `value ± q_mass` with `q_mass` the
`mass` quantile of `|noise|`. A refused cell charges nothing.

In [ ]:
out = release(cell, policy, ledger, release_id="r-laplace", epsilon=0.5, clip=(0.0, 1.0), seed=0, mechanism="laplace")
assert isinstance(out, tuple)
rel, after = out
print(f"{rel.mechanism}: value {rel.value:.4f}  {rel.interval}  true clipped mean {np.mean(np.clip(cell.values, 0, 1)):.4f}")
print(f"  n={rel.n} clipped={rel.n_clipped} sensitivity={rel.sensitivity:.4f} scale={rel.noise_scale:.4f} seed={rel.seed} ε={rel.epsilon} δ={rel.delta}")
print("  ledger after:", after.used, "remaining", after.remaining)

g = release(cell, policy, after, release_id="r-gaussian", epsilon=0.4, clip=(0.0, 1.0), seed=0, mechanism="gaussian", delta=1e-6)
assert isinstance(g, tuple)
rel_g: Release = g[0]
print(f"{rel_g.mechanism}: value {rel_g.value:.4f}  σ={rel_g.noise_scale:.4f}  δ={rel_g.delta} | {rel_g.detail['noise_scale']}")
print("  ledger after both:", g[1].used, "| round-trips:", Spec.from_json(rel_g.to_json()) == rel_g)

refused = release(small, policy, ledger, release_id="r-small", epsilon=0.1, clip=(0.0, 1.0), seed=0)
print("refused cell:", type(refused).__name__, "-", refused.reason)
broke = release(cell, policy, g[1], release_id="r-late", epsilon=0.5, clip=(0.0, 1.0), seed=0)
print("over budget: ", type(broke).__name__, "-", broke.reason[:50], "...")

## `orthogonal_split` and `carry_forward`

`orthogonal_split` divides an ε equally among parts under sequential composition (the parts sum
to ε exactly). `carry_forward` re-uses a previous release at no cost when the cell's contributor
churn — the `jaccard_distance` between the old and new contributor sets — is below a threshold;
above it the result is `Blocked` naming the churn and a new, charged release is required.

In [ ]:
split: EpsilonSplit = orthogonal_split(1.0, 3)
print(split.epsilons, "| sum:", sum(split.epsilons), "|", split.composition)

same = cell_from_records(records + [records[0].model_copy(update={"study": "s9", "contributor": "c5"})], name="next period")
churn = jaccard_distance(frozenset(rel.contributors), same.contributors)
carried = carry_forward(rel, same, policy, churn_threshold=0.25)
assert isinstance(carried, Release)
print(f"churn {churn:.3f} ≤ 0.25: carried from {carried.carried_from!r}, value unchanged {carried.value == rel.value}")

turnover = Cell(records=(("x0", 0.4), ("x1", 0.5), ("x2", 0.6), ("c0", 0.42)), name="new contributors")
blocked = carry_forward(rel, turnover, policy, churn_threshold=0.25)
assert isinstance(blocked, Blocked)
print("churn", round(jaccard_distance(frozenset(rel.contributors), turnover.contributors), 3), "->", blocked.reason)